In [33]:
from astroquery.mast import Catalogs, Observations
import pandas as pd
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
import lightkurve as lk
import pickle
from astropy.utils.data import clear_download_cache
from astropy.io import fits

In [35]:
data_folder = '/Users/philvanlane/Documents/lc_ae/data/'
lcs = pd.read_csv(data_folder + 'mocadb_add_old_LCs.csv')
lcs = lcs[lcs['TIC_ID'].astype(str) == lcs['target_name'].astype(str)]


In [37]:
lcs['sector'] = lcs['mission'].str[12:100].astype(int)

In [38]:
lcs

,TIC_ID,mission,author,exptime,year,target_name,sector
0,4374147,TESS Sector 20,SPOC,120.0 s,2019,4374147,20
1,17901654,TESS Sector 20,SPOC,120.0 s,2019,17901654,20
2,97314759,TESS Sector 44,SPOC,120.0 s,2021,97314759,44
3,97314759,TESS Sector 45,SPOC,120.0 s,2021,97314759,45
4,97314759,TESS Sector 46,SPOC,120.0 s,2021,97314759,46
...,...,...,...,...,...,...,...
145,130931009,TESS Sector 92,SPOC,120.0 s,2025,130931009,92
146,130289670,TESS Sector 92,SPOC,120.0 s,2025,130289670,92
147,130631147,TESS Sector 54,SPOC,120.0 s,2022,130631147,54
148,130631147,TESS Sector 92,SPOC,120.0 s,2025,130631147,92


### Download lightcurves and save them to file

In [39]:
all_lc_data = {}

In [30]:
lcs['TIC_ID'].values[805]

178872239

In [5]:
with open(data_folder + "lc_data.pickle", "rb") as file:
    all_lc_data = pickle.load(file)

EOFError: Ran out of input

In [41]:
for i in range(len(lcs)):
    if i % 50 == 0:
        print(f"Processing light curve {i}/{len(lcs)}")
        # with open(data_folder + "lc_data.pickle", "wb") as file:
        #     pickle.dump(all_lc_data, file)
    tic_id = lcs['TIC_ID'].values[i]
    sector = lcs['sector'].values[i]
    key = f'{tic_id}_{sector}'
    if key in all_lc_data:
        continue
    res = lk.search_lightcurve(f"TIC {tic_id}", mission='TESS', sector=sector, author='SPOC', exptime=120)
    filt_res = res[(res.table['target_name'] == str(tic_id))]
    if len(filt_res) > 0:
        try:
            lc = filt_res[0].download()
            time = lc['time'].value
            flux = lc['pdcsap_flux'].value.filled(np.nan)
            flux_err = lc['pdcsap_flux_err'].value.filled(np.nan)

            # Metadata
            camera = lc.meta['CAMERA']
            ccd = lc.meta['CCD']


            # Mask out non-finite values
            mask = np.isfinite(flux) & np.isfinite(time)
            flux = flux[mask]
            time = time[mask]
            flux_err = flux_err[mask]

            # Normalize flux values
            flux_norm = (flux - np.mean(flux)) / np.std(flux)
            flux_err_norm = flux_err / np.std(flux)
            med_flux_err = np.nanmedian(flux_err_norm)
            mean_flux = np.mean(flux)
            std_flux = np.std(flux)

            # Saving to dict
            rec = {
                'TIC_ID': tic_id,
                'sector': sector,
                'time': time,
                'flux': flux_norm,
                'flux_err': flux_err_norm,
                'metadata': {
                    'TIC_ID': tic_id,
                    'sector': sector,
                    'camera': camera,
                    'ccd': ccd,
                    'mean_flux': mean_flux,
                    'std_flux': std_flux
                }
            }
            all_lc_data[key] = rec
        except Exception as e:
            print(f"⚠️ Download failed for TIC {tic_id}, sector {sector}: {e}\n")
    else:
        print(f"⚠️ No light curve data found for TIC {tic_id} in sector {sector}\n")
    clear_download_cache()

Processing light curve 0/150
Processing light curve 50/150
Processing light curve 100/150


In [42]:
len(all_lc_data)

150

In [23]:
with open(data_folder + "lc_data.pickle", "wb") as file:
    pickle.dump(all_lc_data, file)

In [26]:
all_lc_data['72437047_49']

{'TIC_ID': 72437047,
 'sector': 49,
 'time': array([2640.10594291, 2640.10733184, 2640.10872077, ..., 2664.31727761,
        2664.31866648, 2664.32005535]),
 'flux': array([-0.78598976, -0.7598669 , -0.5636896 , ...,  0.10474741,
         0.13035803, -0.20898262], dtype=float32),
 'flux_err': array([0.12001014, 0.12000565, 0.12000702, ..., 0.11951794, 0.11951654,
        0.11950621], dtype=float32),
 'metadata': {'TIC_ID': 72437047,
  'sector': 49,
  'camera': 1,
  'ccd': 1,
  'mean_flux': 4744697.0,
  'std_flux': 1952.3156}}

In [27]:
import pandas as pd
import pickle

data_folder = '/Users/philvanlane/Documents/lc_ae/exop_hosts/'

# Load metadata
meta = pd.read_csv(data_folder + 'host_all_metadata.csv')
meta = meta.set_index('TIC_ID')
fields = ['Tmag', 'parallax', 'parallax_error', 'G0', 'G0_err', 'BPRP0', 'BPRP0_err']

# Load pickle
with open(data_folder + 'lc_data.pickle', 'rb') as f:
    all_lc_data = pickle.load(f)

# Verify every TIC in the pickle has a row in the CSV
tic_ids_in_pickle = {v['TIC_ID'] for v in all_lc_data.values()}
missing = tic_ids_in_pickle - set(meta.index)
if missing:
    print(f"WARNING: {len(missing)} TIC IDs missing from CSV: {missing}")
else:
    print(f"All {len(tic_ids_in_pickle)} unique TIC IDs found in CSV ✓")

# Add fields to each record's metadata dict
not_found = set()
for key, rec in all_lc_data.items():
    tic = rec['TIC_ID']
    if tic in meta.index:
        for field in fields:
            rec['metadata'][field] = meta.at[tic, field]
    else:
        not_found.add(tic)

if not_found:
    print(f"WARNING: {len(not_found)} TIC IDs not found during update: {not_found}")
else:
    print("All records updated successfully ✓")

# Verify one record
sample_key = next(iter(all_lc_data))
print(f"\nSample metadata for {sample_key}:")
print(all_lc_data[sample_key]['metadata'])

# Save
with open(data_folder + 'lc_data.pickle', 'wb') as f:
    pickle.dump(all_lc_data, f)
print("\nSaved.")



Sample metadata for 72437047_49:
{'TIC_ID': 72437047, 'sector': 49, 'camera': 1, 'ccd': 1, 'mean_flux': 4744697.0, 'std_flux': 1952.3156, 'Tmag': 3.8379, 'parallax': 10.203968527196396, 'parallax_error': 0.13647683, 'G0': 4.43601141295932, 'G0_err': 0.0100501914170995, 'BPRP0': 1.186981694489479, 'BPRP0_err': 0.014432842281944}

Saved.


In [30]:
import pickle
import numpy as np

with open('/Users/philvanlane/Documents/lc_ae/exop_hosts/lc_data.pickle', 'rb') as f:
    all_lc_data = pickle.load(f)

# Find records with suspicious lengths
MAX_EXPECTED = 20160  # 28 days at 2-min cadence
outliers = {k: v for k, v in all_lc_data.items() if len(v['flux']) > MAX_EXPECTED}

print(f"Records > {MAX_EXPECTED} points: {len(outliers)}")
print()

for key, rec in sorted(outliers.items(), key=lambda x: -len(x[1]['flux']))[:20]:
    t = rec['time']
    duration_days = (t[-1] - t[0]) if len(t) > 1 else 0
    cadence_s = np.median(np.diff(t)) * 86400 if len(t) > 1 else 0
    print(f"{key:25s}  n={len(rec['flux']):6d}  duration={duration_days:.1f}d  median_cadence={cadence_s:.1f}s")


Records > 20160 points: 298

32001057_97                n= 30617  duration=52.4d  median_cadence=120.0s
29781292_97                n= 30617  duration=52.4d  median_cadence=120.0s
38828538_97                n= 30617  duration=52.4d  median_cadence=120.0s
179582003_97               n= 30617  duration=52.4d  median_cadence=120.0s
220017951_97               n= 30431  duration=52.4d  median_cadence=120.0s
220022156_97               n= 30431  duration=52.4d  median_cadence=120.0s
166853853_97               n= 30431  duration=52.4d  median_cadence=120.0s
200723869_97               n= 30431  duration=52.4d  median_cadence=120.0s
166739520_97               n= 30431  duration=52.4d  median_cadence=120.0s
166836920_97               n= 30431  duration=52.4d  median_cadence=120.0s
1309019_98                 n= 30337  duration=57.4d  median_cadence=120.0s
1167538_98                 n= 30337  duration=57.4d  median_cadence=120.0s
170729775_98               n= 30337  duration=57.4d  median_cadence=120

In [32]:
import pickle
import h5py
import numpy as np
from pathlib import Path

data_folder = '/Users/philvanlane/Documents/lc_ae/exop_hosts/'

with open(data_folder + 'lc_data.pickle', 'rb') as f:
    all_lc_data = pickle.load(f)

# Exclude records with any missing metadata
required_fields = ['Tmag', 'parallax', 'parallax_error', 'G0', 'G0_err', 'BPRP0', 'BPRP0_err']
valid_keys = sorted([
    k for k, v in all_lc_data.items()
    if all(f in v['metadata'] and not np.isnan(v['metadata'][f]) for f in required_fields)
])
excluded = len(all_lc_data) - len(valid_keys)
print(f"Records: {len(valid_keys)} valid, {excluded} excluded (missing metadata)")

max_len = max(len(all_lc_data[k]['flux']) for k in valid_keys)
print(f"Global max length: {max_len}")

N = len(valid_keys)
flux_arr     = np.zeros((N, max_len), dtype=np.float32)
flux_err_arr = np.zeros((N, max_len), dtype=np.float32)
time_arr     = np.zeros((N, max_len), dtype=np.float32)
lengths      = np.zeros(N, dtype=np.int32)

meta_arrays = {
    'tic':                        np.zeros(N, dtype=np.int64),
    'sector':                     np.zeros(N, dtype=np.int64),
    'camera':                     np.zeros(N, dtype=np.int32),
    'ccd':                        np.zeros(N, dtype=np.int32),
    'cadence_s':                  np.full(N, 120.0, dtype=np.float64),
    'mean_flux':                  np.zeros(N, dtype=np.float32),
    'std_flux':                   np.zeros(N, dtype=np.float32),
    'Tmag':                       np.zeros(N, dtype=np.float64),
    'dr3_parallax_zpt_corrected': np.zeros(N, dtype=np.float64),
    'parallax_error':             np.zeros(N, dtype=np.float64),
    'G_0':                        np.zeros(N, dtype=np.float64),
    'e_G_0':                      np.zeros(N, dtype=np.float64),
    'BPRP0':                      np.zeros(N, dtype=np.float64),
    'e_BPRP0':                    np.zeros(N, dtype=np.float64),
    'duration':                   np.zeros(N, dtype=np.float64),
    'n_points':                   np.zeros(N, dtype=np.int64),
    'med_flux_error':             np.zeros(N, dtype=np.float32),
}

nan_fill_count = 0
for i, key in enumerate(valid_keys):
    rec = all_lc_data[key]
    L = len(rec['flux'])

    flux_err = rec['flux_err'].copy()
    med_flux_err = float(np.nanmedian(flux_err))
    nan_mask = ~np.isfinite(flux_err)
    nan_fill_count += int(nan_mask.sum())
    flux_err[nan_mask] = med_flux_err

    flux_arr[i, :L]     = rec['flux']
    flux_err_arr[i, :L] = flux_err
    time_arr[i, :L]     = rec['time']
    lengths[i] = L

    m = rec['metadata']
    meta_arrays['tic'][i]                        = m['TIC_ID']
    meta_arrays['sector'][i]                     = m['sector']
    meta_arrays['camera'][i]                     = m['camera']
    meta_arrays['ccd'][i]                        = m['ccd']
    meta_arrays['mean_flux'][i]                  = m['mean_flux']
    meta_arrays['std_flux'][i]                   = m['std_flux']
    meta_arrays['Tmag'][i]                       = m['Tmag']
    meta_arrays['dr3_parallax_zpt_corrected'][i] = m['parallax']
    meta_arrays['parallax_error'][i]             = m['parallax_error']
    meta_arrays['G_0'][i]                        = m['G0']
    meta_arrays['e_G_0'][i]                      = m['G0_err']
    meta_arrays['BPRP0'][i]                      = m['BPRP0']
    meta_arrays['e_BPRP0'][i]                    = m['BPRP0_err']
    meta_arrays['duration'][i]                   = float(rec['time'][-1] - rec['time'][0]) if L > 1 else 0.0
    meta_arrays['n_points'][i]                   = L
    meta_arrays['med_flux_error'][i]             = med_flux_err

print(f"Filled {nan_fill_count:,} NaN flux_err values with per-sector medians")

output_path = '/Users/philvanlane/Documents/lc_ae/data/timeseries_exop.h5'
with h5py.File(output_path, 'w') as f:
    f.create_dataset('flux',     data=flux_arr,     compression='gzip', compression_opts=4)
    f.create_dataset('flux_err', data=flux_err_arr, compression='gzip', compression_opts=4)
    f.create_dataset('time',     data=time_arr,     compression='gzip', compression_opts=4)
    f.create_dataset('length',   data=lengths)
    grp = f.create_group('metadata')
    for name, arr in meta_arrays.items():
        grp.create_dataset(name, data=arr)

size_gb = Path(output_path).stat().st_size / 1e9
print(f"Saved {N} records → {output_path} ({size_gb:.2f} GB)")


Records: 19969 valid, 66 excluded (missing metadata)
Global max length: 30617
Filled 0 NaN flux_err values with per-sector medians
Saved 19969 records → /Users/philvanlane/Documents/lc_ae/data/timeseries_exop.h5 (2.58 GB)
